[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/05_%E5%AE%9F%E8%B7%B5_BtoB%E3%83%9E%E3%83%BC%E3%82%B1/04_%E9%A1%A7%E5%AE%A2%E3%82%B9%E3%82%B3%E3%82%A2%E3%83%AA%E3%83%B3%E3%82%B0.ipynb)

In [ ]:
# === ① セットアップ（最初に実行してください）===
import pandas as pd               # 表データ
import numpy as np                # 数値計算
import matplotlib.pyplot as plt   # グラフ描画
import os
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の文字化け防止
# ローカルに data/ が無ければ公開リポジトリ(raw)から読み込む
RAW = 'https://raw.githubusercontent.com/Carlo-Broschi/statistics-python-for-students/main/data/'
def load(name):
    p = f'../data/{name}'
    return pd.read_csv(p) if os.path.exists(p) else pd.read_csv(RAW + name)
print('準備OK')

# 実践マーケ-04. 顧客スコアリング（標準化・偏差値）

**舞台設定**：優良顧客を見つけたい。でも「購入回数」「累計売上」「最終購入からの日数」は単位もスケールもバラバラ。そのまま足すと、桁の大きい売上だけで決まってしまいます。

**使う統計（読む=3級／本質=3〜2級）**：標準化（Zスコア）・偏差値・正規分布での順位づけ。

### 使うデータ：`btob_customers.csv`（架空のBtoB顧客320件）
1行＝顧客1件。`購入回数`・`累計売上`・`最終購入日` から優良度を測ります。

| 顧客ID | 業界 | 購入回数 | 累計売上 | 最終購入日 |
|---|---|---|---|---|
| C0124 | 医療 | 4 | 3,843,000 | 2025-11-25 |

## この章でできるようになること
- なぜ「単位がちがう指標」をそのまま足してはいけないかを説明できる
- Zスコア `z=(x−平均)/標準偏差` で指標を共通のものさしに揃えられる
- 偏差値・正規分布で「上位◯%の優良客ライン」を引ける

> **前提**：統計3級（標準化・正規分布）　/　**所要**：約20分　/　**レベル**：実践（読む3級・本質3〜2級）

## 1. データを用意して Recency（最終購入からの日数）を作る

In [ ]:
cust = load('btob_customers.csv')
基準日 = pd.Timestamp('2026-01-01')                                   # 分析の基準日
cust['Recency'] = (基準日 - pd.to_datetime(cust['最終購入日'])).dt.days  # 最終購入からの日数（小さいほど新しい）
cust[['顧客ID','購入回数','累計売上','Recency']].head()

## 2. なぜ標準化？ 単位がちがうと足せない

`累計売上`(数百万) と `購入回数`(1桁) をそのまま足すと、売上の桁が大きすぎて購入回数は無視されます。**標準化（Zスコア）**で「平均からSD何個分か」に直すと、どの指標も平均0・標準偏差1にそろい、対等に比べられます。

In [ ]:
cols = ['購入回数','累計売上','Recency']
Z = (cust[cols] - cust[cols].mean()) / cust[cols].std(ddof=0)   # Zスコア=(x−平均)/標準偏差
print('標準化後の平均（≈0）:', Z.mean().round(3).to_dict())
print('標準化後の標準偏差（=1）:', Z.std(ddof=0).round(3).to_dict())
Z.head()

## 3. 合成スコアを作って顧客をランキング

「よく買う・たくさん使う・最近来た」ほど優良です。`購入回数`と`累計売上`は大きいほど良い(＋)、`Recency`は小さいほど良い(−)ので、符号をそろえて足します。

In [ ]:
cust['スコア'] = Z['購入回数'] + Z['累計売上'] - Z['Recency']   # Recencyは小さいほど良いのでマイナス
ranked = cust.sort_values('スコア', ascending=False)
print('▼ 優良トップ5'); print(ranked[['顧客ID','業界','購入回数','累計売上','Recency','スコア']].head().round(2))

## 4. 偏差値と「上位5%の優良客ライン」（正規分布）

スコアを **偏差値**（平均50・SD10）に直すと直感的です。さらにスコアが正規分布に従うと近似すれば、`ppf` で「上位5%の境目」を引けます（ここは2級の母集団の考え方）。

In [ ]:
s = cust['スコア']
cust['偏差値'] = 50 + 10*(s - s.mean())/s.std(ddof=0)        # 偏差値=50+10z
from scipy import stats
line = stats.norm.ppf(0.95, s.mean(), s.std(ddof=0))         # 上位5%の境目（正規近似）
vip = cust[cust['スコア'] >= line]
print(f'上位5%ラインのスコア: {line:.2f}  /  該当した優良客: {len(vip)}人（全{len(cust)}人）')
fig, ax = plt.subplots(figsize=(7,4))
ax.hist(s, bins=30, alpha=.7); ax.axvline(line, color='crimson', ls='--', label='上位5%ライン')
ax.set_xlabel('合成スコア'); ax.set_ylabel('人数'); ax.legend(); ax.set_title('顧客スコアの分布'); plt.show()

## まとめ
- 単位のちがう指標は **標準化（Zスコア）** で平均0・SD1にそろえてから合成する。
- 符号をそろえて足せば、複数指標を1つの **優良度スコア** にまとめられる。
- 偏差値や正規分布の `ppf` で「上位◯%」の線を引ける。

> 標準化は、単位のちがう指標を比べる多くの分析（相関・回帰など）の下ごしらえになります。

> **よくある間違い**
> - 標準化しても **分布の形は変わりません**（正規分布になるわけではない）。上位%の話は近似。
> - Zスコアは平均・SDを使うので **外れ値に弱い**。極端な大口客がいると全体が歪みます（中央値・IQRの活用も検討）。

## 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます。

In [ ]:
# 採点用の関数（答え合わせに使うだけ・答えは表示しません）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
# Q1: x=20, 平均10, 標準偏差5 のときのZスコア (x−平均)/標準偏差 を ans に
ans = None   # 例: (20-10)/5
_check('Q1 Zスコア', ans, (20-10)/5)

In [ ]:
# Q2: 標準化した後のデータの平均は？ を ans に
ans = None   # ヒント: 標準化の定義から決まる値
_check('Q2 標準化後の平均', ans, 0)

In [ ]:
# Q3: Zスコアが +1 の人の偏差値（50+10z）を ans に
ans = None   # 例: 50 + 10*1
_check('Q3 偏差値', ans, 50 + 10*1)

---
## ワークシート課題

**課題1.** `業界` ごとにグループ分けして、業界内での偏差値を計算してみよう（`groupby('業界')` で平均・SDを業界別に）。全体の偏差値と何が変わる？

**課題2.** 合成スコアの重みを変えて、`累計売上` を2倍重視したスコアを作ろう。トップ5は入れ替わる？

**課題3.**（考察）外れ値に強くするには、Zスコアの代わりに「中央値とIQR」で標準化する手もある。どんな顧客データのときに有効か、1〜2行で説明しよう。

> **解答例は別ページ**（ネタバレ防止）**[解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/05_%E5%AE%9F%E8%B7%B5_BtoB%E3%83%9E%E3%83%BC%E3%82%B1/04_%E9%A1%A7%E5%AE%A2%E3%82%B9%E3%82%B3%E3%82%A2%E3%83%AA%E3%83%B3%E3%82%B0.md)**
>
> 自分で解いてから開きましょう。

## 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 標準化（Zスコア） | (x−平均)/標準偏差。平均0・SD1にそろえる |
| 偏差値 | 50+10z。平均50・SD10に直した指標 |
| Recency | 最終購入からの経過日数（小さいほど新しい） |
| ppf | 確率→値。`norm.ppf(0.95,μ,σ)`＝上位5%の境目 |

```python
Z = (df[cols]-df[cols].mean())/df[cols].std(ddof=0)   # 標準化
偏差値 = 50 + 10*Z
from scipy import stats; stats.norm.ppf(0.95, mu, sd)  # 上位5%ライン
```

## 完了ログ

このノートを終えたら下のセルを実行すると、学習の完了が記録されます。
**学習者コードは最初に一度だけ設定**すれば、以降は全ノートで自動送信されます（名前の再入力は不要）。

- Colab：左の鍵アイコン（シークレット）で `STUDENT_CODE` に配布コードを登録（1回だけ）
- ローカル：環境変数 `STUDENT_CODE` を設定（または初回に画面入力すると保存され、次回から不要）

In [ ]:
# === 完了ログ ===  学習者コードは最初に一度だけ設定すれば、全ノートで自動送信されます。
import os, datetime, pathlib
try:
    import requests
except ImportError:
    requests = None

def _student_code():
    try:                                          # Colab: 鍵アイコンで STUDENT_CODE を登録
        from google.colab import userdata
        c = userdata.get('STUDENT_CODE')
        if c: return c.strip()
    except Exception:
        pass
    c = os.environ.get('STUDENT_CODE')            # ローカル: 環境変数
    if c: return c.strip()
    p = pathlib.Path.home() / '.student_code'      # 保存ファイル
    if p.exists(): return p.read_text().strip()
    try:                                           # 最後の手段: 入力して保存（次回から不要）
        c = input('学習者コードを入力（配布されたもの）: ').strip()
        if c: p.write_text(c); return c
    except Exception:
        pass
    return ''

CODE = _student_code()
LOG_URL = ""      # 配布時に設定
LOG_TOKEN = ""    # 配布時に設定
NOTEBOOK = "05_実践_BtoBマーケ/04_顧客スコアリング"

if CODE and LOG_URL and requests:
    try:
        requests.post(LOG_URL, json={"token": LOG_TOKEN, "code": CODE, "notebook": NOTEBOOK,
                      "time": datetime.datetime.now().isoformat(timespec="seconds")}, timeout=10)
        print(f"記録しました: {CODE} / {NOTEBOOK}")
    except Exception as e:
        print("記録に失敗しました（URL/ネットワークを確認）:", e)
elif not CODE:
    print("学習者コード未設定。Colabは鍵アイコンで STUDENT_CODE を登録、ローカルは環境変数 STUDENT_CODE を設定してください。")
else:
    print(f"{NOTEBOOK}: LOG_URL/LOG_TOKEN が未設定です（配布時に設定されます）。")